**1) Loading data and saving to the schema:**

In [0]:
import pandas as pd
import requests
login = "konstyantyn"
table = "co_emissions_per_capita"
catalog = "dbr_dev"
url = "https://ourworldindata.org/grapher/co-emissions-per-capita.csv?v=1&csvType=full&useColumnShortNames=true"
#use of try if we have some mistakes
try:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() #status check
    pdf = pd.read_csv(url) 
    df = spark.createDataFrame(pdf)
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{login}") # catalog creation in databricks
    (df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{login}.{table}"))
    print(f" Data and schema was succesfuly saved to {catalog}.{login}.{table}")
except Exception as e:
    print(f"Error: {e}")

**2)Basic Spark DataFrame operations:**

In [0]:
df_raw = spark.read.table("dbr_dev.konstyantyn.co_emissions_per_capita")
print("Schema check:")
df_raw.printSchema()
print("Data check:")
display(df_raw)

In [0]:
#Average emissions per capita inside each country
import pyspark.sql.functions as F
df_filtered = df_raw.select(F.col("entity"), F.col("year"), F.col("emissions_total_per_capita")).filter((F.col("year") >= 2010) & (F.col("emissions_total_per_capita").isNotNull())).groupBy("entity").agg(F.avg("emissions_total_per_capita").alias("avg_emissions_total_per_capita")).orderBy(F.col("avg_emissions_total_per_capita"))
print("Schema check:")
df_filtered.printSchema()
print("Filtered data check:")
display(df_filtered)

Databricks visualization. Run in Databricks to view.

In [0]:
# Poland emissions per capita
import pyspark.sql.functions as F
df_filtered2 = df_raw.select(F.col("entity"), F.col("year"), F.col("emissions_total_per_capita")).filter(F.col("entity") == "Poland").filter((F.col("year") >= 2010) & (F.col("emissions_total_per_capita").isNotNull())).groupBy("entity", "year").agg(F.avg("emissions_total_per_capita").alias("avg_emissions_total_per_capita")).orderBy(F.col("avg_emissions_total_per_capita")).orderBy(F.col("avg_emissions_total_per_capita"))
display(df_filtered2)

Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 countries with the highest emissions per capita
import pyspark.sql.functions as F
df_filtered3 = df_raw.select(F.col("entity"), F.col("year"), F.col("emissions_total_per_capita")).filter(F.col("entity").isNotNull()).filter((F.col("year") >= 2010) & (F.col("emissions_total_per_capita").isNotNull())).groupBy("entity").agg(F.max("emissions_total_per_capita").alias("max_emissions_total_per_capita")).orderBy(F.col("max_emissions_total_per_capita").desc()).limit(10)
display(df_filtered3)


Databricks visualization. Run in Databricks to view.

**3)API data json:**

In [0]:
import requests
import json
login = "konstyantyn"
api_table = "co2_api_metadata"
catalog = "dbr_dev"
api_url = "https://ourworldindata.org/grapher/co-emissions-per-capita.metadata.json?v=1&csvType=full&useColumnShortNames=true"

try:
    print("Downloading JSON from API Our World in Data...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # downolading data
    response = requests.get(api_url, headers=headers, timeout=15)
    response.raise_for_status()
    
    # loading json to python
    json_data = json.loads(response.text)
    
    # create Spark DataFrame api
    df_api = spark.createDataFrame([(response.text,)], ["raw_response"])
    # save to delta
    (df_api.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(f"{catalog}.{login}.{api_table}"))
    
    print(f"Success! API saved to the table: {catalog}.{login}.{api_table}")
    display(df_api)

except Exception as e:
    print(f"Error API related: {e}")

**Additional task to "explain Delta Lake benefits": Data lake make easy to move with different data types, create tables and schemas and work with them, easy to create visual effects and has fast high performance.**